# Internalizing MCP Tool Knowledge in Small LLMs
## End-to-End: Baseline → Generalist Fine-Tuning → Specialist Models → Comparison

**HPML Group 20 — Columbia University, Spring 2026**

Pipeline on **1x H100 (80 GB)**:

1. **Setup & data** — clone repo, load datasets, create proper train/test split (no leakage)
2. **Baseline evaluation** — test the base model in informed (with tool descriptions) and blind (no descriptions) mode
3. **Generalist fine-tuning** — train one model on ALL tool knowledge + plans
4. **Specialist fine-tuning** — train per-domain models (IoT, FMSR, TSFM, WorkOrder)
5. **Evaluation** — compare generalist vs specialists on held-out scenarios, measure token savings

**Research question:** *Can Gemma internalize tool descriptions to plan without them in the prompt? Do domain specialists outperform a generalist?*

## 0. Setup & Installation CHECKED

In [1]:
!pip install -U transformers>=5.5.0 peft>=0.13.0 trl>=0.12.0
!pip install -U bitsandbytes>=0.44.0 accelerate>=1.0.0
!pip install -q datasets evaluate rouge-score
!pip install -q pandas matplotlib seaborn tqdm
!pip install -q sentencepiece protobuf

import transformers
print(f"transformers version: {transformers.__version__}")
print("All packages installed.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
transformers version: 5.7.0
All packages installed.


In [2]:
import os, json, re, time, random, warnings, inspect
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
from copy import deepcopy
import torch
import numpy as np
from tqdm.auto import tqdm
from transformers import TrainingArguments

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({mem:.1f} GB)")

model_name = "google/gemma-2b"
lora_r = 8
learning_rate = 2e-5
num_epochs = 3
batch_size = 2

train_dataset = [{"text": "sample"}] * 100

training_args = TrainingArguments(
    output_dir="./results",
    report_to="wandb",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=2,
    num_train_epochs=num_epochs,
    learning_rate=learning_rate,
    fp16=torch.cuda.is_available(),
)

print("Setup complete")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB (85.1 GB)
Setup complete


## 1. Configuration CHECKED

In [ ]:
import os
import torch
from datetime import datetime
import wandb

# =========================
# MODE (FULL RUN ONLY)
# =========================
LIGHT_MODE = False  # kept for compatibility only

# =========================
# MODEL
# =========================
MODEL_ID = "google/gemma-4-E4B-it"
LOAD_IN_8BIT = True

# =========================
# HF TOKEN
# =========================
if not HF_TOKEN:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("Loaded HF_TOKEN from Colab secrets")
    except Exception:
        print(f"WARNING: Set HF_TOKEN. Accept license at https://huggingface.co/{MODEL_ID}")

# =========================
# REPO
# =========================
REPO_URL = "https://github.com/YuvalShemla/hpml-2026-project.git"
REPO_DIR = "/content/hpml-2026-project"

# =========================
# LORA (FIXED FOR FULL RUN)
# =========================
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"

# =========================
# HARDWARE-AWARE SETTINGS
# =========================
_gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 40

if _gpu_mem_gb > 60:
    MAX_SEQ_LENGTH = 2048
    PER_DEVICE_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 4
else:
    MAX_SEQ_LENGTH = 1024
    PER_DEVICE_BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 8

# =========================
# TRAINING PARAMS (TASK SPEC)
# =========================
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"
BF16 = True

# =========================
# GENERATION / EVAL
# =========================
NUM_HELD_OUT = 50
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.1
TOP_P = 0.9

# =========================
# DATA
# =========================
MAX_TRAIN_EXAMPLES = None  # full dataset

# =========================
# OUTPUT
# =========================
OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# RUN NAME
# =========================
run_name = f"gemma4b-full-r{LORA_R}-bs{PER_DEVICE_BATCH_SIZE}"

# =========================
# W&B (CLEAN INIT)
# =========================
import wandb

wandb.login()

# ensure previous runs are closed (important in notebooks)
if wandb.run is not None:
    wandb.finish()

run_name = f"{MODEL_ID.split('/')[-1]}-r{LORA_R}-bs{PER_DEVICE_BATCH_SIZE}-{int(time.time())}"

wandb.init(
    project="hpml-group20-assetops",
    name=run_name,
    job_type="train",
    reinit=True,  # 🔑 important for multiple runs in same notebook
    config={
        "model_id": MODEL_ID,
        "load_in_8bit": LOAD_IN_8BIT,

        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,

        "max_seq_length": MAX_SEQ_LENGTH,
        "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,

        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "lr_scheduler": LR_SCHEDULER,
        "bf16": BF16,

        "num_held_out": NUM_HELD_OUT,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,

        "max_train_examples": MAX_TRAIN_EXAMPLES,

        "gpu_mem_gb": float(_gpu_mem_gb),
        "timestamp": datetime.now().isoformat()
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:


KeyboardInterrupt: 

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,

    bf16=BF16,
    logging_steps=10,
    save_strategy="epoch",

    report_to="wandb",
    run_name=run_name,

    optim="paged_adamw_8bit",  # important for QLoRA
    gradient_checkpointing=True,

    max_grad_norm=0.3,
)

## 2. Clone Repository & Load Data CHECKED

In [ ]:
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

datasets_dir = Path(REPO_DIR) / "benchmark" / "generate_data" / "datasets"
ds_tool = load_jsonl(datasets_dir / "tool_knowledge.jsonl")
ds_plan = load_jsonl(datasets_dir / "planning.jsonl")
ds_exec = load_jsonl(datasets_dir / "execution.jsonl")

gold_path = Path(REPO_DIR) / "benchmark" / "baseline_tests" / "gemini_flash_informed_results.json"
with open(gold_path) as f:
    gold_plans_raw = json.load(f)
gold_plans = {g["id"]: g for g in gold_plans_raw if g.get("plan_steps", 0) > 0}

print(f"Tool knowledge: {len(ds_tool)}, Planning: {len(ds_plan)}, Execution: {len(ds_exec)}")
print(f"Gold plans: {len(gold_plans)}")

## 3. Train/Test Split (No Data Leakage) CHECKED

We hold out scenarios for evaluation and remove them AND their paraphrases from training.
The split is stratified by scenario type (IoT, FMSA, TSFM, Workorder, multiagent).

In [ ]:
from datasets import load_dataset

hf_ds = load_dataset("ibm-research/AssetOpsBench", "scenarios")
hf_scenarios = [dict(row) for row in hf_ds["train"]]

# Build full eval candidate list (scenarios with gold plans)
eval_candidates = []
for sc in hf_scenarios:
    if sc["id"] in gold_plans:
        eval_candidates.append({
            "id": sc["id"],
            "question": sc["text"],
            "type": sc["type"],
            "category": sc["category"],
            "gold_plan": gold_plans[sc["id"]]["response"],
            "gold_steps": gold_plans[sc["id"]]["plan_steps"],
            "gold_agents": gold_plans[sc["id"]].get("agents_used", []),
            "gold_tools": gold_plans[sc["id"]].get("tools_used", []),
        })

# Stratified split: hold out NUM_HELD_OUT scenarios, balanced by type
random.shuffle(eval_candidates)
by_type = defaultdict(list)
for sc in eval_candidates:
    by_type[sc["type"]].append(sc)

eval_scenarios = []
per_type_quota = max(1, NUM_HELD_OUT // len(by_type))
for t, scenarios in by_type.items():
    n = min(per_type_quota, len(scenarios))
    eval_scenarios.extend(scenarios[:n])

# Fill remaining quota from largest types
remaining = NUM_HELD_OUT - len(eval_scenarios)
eval_ids = {s["id"] for s in eval_scenarios}
for t in sorted(by_type.keys(), key=lambda t: -len(by_type[t])):
    for sc in by_type[t]:
        if remaining <= 0:
            break
        if sc["id"] not in eval_ids:
            eval_scenarios.append(sc)
            eval_ids.add(sc["id"])
            remaining -= 1

eval_questions_lower = {sc["question"].strip().lower() for sc in eval_scenarios}

print(f"Held-out eval scenarios: {len(eval_scenarios)}")
print(f"Type distribution: {dict(Counter(s['type'] for s in eval_scenarios))}")

# Remove held-out scenarios AND their paraphrases from training
# A paraphrase shares the same plan response as the gold scenario it paraphrases from
held_out_plans = set()
for sc in eval_scenarios:
    held_out_plans.add(sc["gold_plan"].strip())

def is_leaked(example):
    q = example["messages"][0]["content"].strip().lower()
    # Direct question match
    if q in eval_questions_lower:
        return True
    return False

clean_tool = [d for d in ds_tool if not is_leaked(d)]
clean_plan = [d for d in ds_plan if not is_leaked(d)]
clean_exec = [d for d in ds_exec if not is_leaked(d)]

print(f"\nTraining data after removing held-out:")
print(f"  Tool knowledge: {len(clean_tool)} (was {len(ds_tool)})")
print(f"  Planning: {len(clean_plan)} (was {len(ds_plan)})")
print(f"  Execution: {len(clean_exec)} (was {len(ds_exec)})")

# Also remove paraphrases that share the exact same plan as held-out scenarios
before_para = len(clean_plan)
clean_plan = [d for d in clean_plan if d["messages"][1]["content"].strip() not in held_out_plans]
print(f"  Planning after removing shared-plan paraphrases: {len(clean_plan)} (removed {before_para - len(clean_plan)} more)")

total_train = len(clean_tool) + len(clean_plan) + len(clean_exec)
print(f"  Total clean training: {total_train}")

## 4. Evaluation Framework CHECKED

Metrics: plan format validity, agent/tool correctness, Agent-Tool pair F1, ROUGE-L.

In [ ]:
from evaluate import load as load_metric
rouge_metric = load_metric("rouge")

VALID_AGENTS = {"IoTAgent", "FMSRAgent", "TSFMAgent", "Utilities", "WorkOrderAgent", "VibrationAgent", "none"}
VALID_TOOLS = {
    "sites", "assets", "sensors", "history",
    "get_failure_modes", "get_failure_mode_sensor_mapping",
    "get_ai_tasks", "get_tsfm_models", "run_tsfm_forecasting",
    "run_tsfm_finetuning", "run_tsad", "run_integrated_tsad",
    "json_reader", "current_date_time", "current_time_english",
    "get_work_orders", "get_preventive_work_orders", "get_corrective_work_orders",
    "get_events", "get_failure_codes", "get_work_order_distribution",
    "predict_next_work_order", "analyze_alert_to_failure",
    "get_vibration_data", "list_vibration_sensors", "compute_fft_spectrum",
    "compute_envelope_spectrum", "assess_vibration_severity",
    "calculate_bearing_frequencies", "list_known_bearings", "diagnose_vibration",
    "none",
}

TOOL_DESCRIPTIONS = """Available Agents and Tools:

IoTAgent: IoT telemetry data access.
  - sites(): List all available IoT sites.
  - assets(site_name): List all asset IDs for a given site.
  - sensors(site_name, asset_id): List sensor names for an asset.
  - history(site_name, asset_id, start, final?): Fetch historical sensor readings.

FMSRAgent: Failure mode analysis and sensor relevance.
  - get_failure_modes(asset_name): Return known failure modes.
  - get_failure_mode_sensor_mapping(asset_name, failure_modes, sensors): Map sensor relevancy.

TSFMAgent: Time-series forecasting and anomaly detection.
  - get_ai_tasks(): List AI task types.
  - get_tsfm_models(): List model checkpoints.
  - run_tsfm_forecasting(dataset_path, timestamp_column, target_columns): Forecast.
  - run_tsfm_finetuning(dataset_path, timestamp_column, target_columns): Fine-tune.
  - run_tsad(dataset_path, tsfm_output_json, timestamp_column, target_columns): Anomaly detection.
  - run_integrated_tsad(dataset_path, timestamp_column, target_columns): E2E forecast + anomaly.

Utilities:
  - json_reader(file_name): Read JSON file.
  - current_date_time(): Current UTC date/time.
  - current_time_english(): Current UTC time readable.

WorkOrderAgent: Work order and maintenance reasoning.
  - get_work_orders(asset_name): Retrieve work orders.
  - get_preventive_work_orders(asset_name): Preventive WOs.
  - get_corrective_work_orders(asset_name): Corrective WOs.
  - get_events(asset_name): Maintenance events/alerts.
  - get_failure_codes(asset_name): Failure codes.
  - get_work_order_distribution(asset_name): WO statistics.
  - predict_next_work_order(asset_name): Predict next WO.
  - analyze_alert_to_failure(asset_name): Alert-failure correlation.
"""

INFORMED_PROMPT = """You are an expert planner for industrial asset operations. Given a question and available tools, produce a structured plan.

{tool_descriptions}

OUTPUT FORMAT:
#Task1: <description>
#Agent1: <agent_name>
#Tool1: <tool_name>
#Args1: {{"arg": "value"}}
#Dependency1: None
#ExpectedOutput1: <what to expect>

Rules: Use ONLY listed agents/tools. Keep plans concise. Use {{step_N}} for dependencies.

QUESTION: {question}
"""

BLIND_PROMPT = """You are an expert planner for industrial asset operations. Produce a structured plan.

OUTPUT FORMAT:
#Task1: <description>
#Agent1: <agent_name>
#Tool1: <tool_name>
#Args1: {{"arg": "value"}}
#Dependency1: None
#ExpectedOutput1: <what to expect>

Rules: Keep plans concise. Use {{step_N}} for dependencies.

QUESTION: {question}
"""


def parse_plan(text):
    """Parse a plan into structured steps with agent, tool, and args."""
    steps = []
    if not text: return steps
    for block in re.split(r'(?=#Task\d+:)', text):
        block = block.strip()
        task_m = re.search(r'#Task(\d+):\s*(.*)', block)
        agent_m = re.search(r'#Agent\d+:\s*(\S+)', block)
        tool_m = re.search(r'#Tool\d+:\s*(\S+)', block)
        args_m = re.search(r'#Args\d+:\s*(.*)', block)
        if task_m:
            # Try to parse args as JSON
            args_raw = args_m.group(1).strip() if args_m else "{}"
            try:
                args_parsed = json.loads(args_raw)
            except (json.JSONDecodeError, TypeError):
                args_parsed = {}
            steps.append({
                "step": int(task_m.group(1)),
                "task": task_m.group(2).strip(),
                "agent": agent_m.group(1).strip() if agent_m else "",
                "tool": tool_m.group(1).strip().rstrip("()") if tool_m else "",
                "args": args_parsed,
                "args_raw": args_raw,
            })
    return steps


def evaluate_plan(gen_text, gold_text):
    """Evaluate generated plan vs gold on routing, args, and text similarity."""
    result = {"has_plan_format": False, "num_steps": 0, "valid_agents": 0,
              "total_agents": 0, "valid_tools": 0, "total_tools": 0,
              "agent_tool_f1": 0.0, "gold_steps": 0, "rouge_l": 0.0,
              "arg_key_f1": 0.0, "arg_value_match": 0.0}
    if "#Task" in gen_text and "#Agent" in gen_text:
        result["has_plan_format"] = True
    gen_steps = parse_plan(gen_text)
    gold_steps = parse_plan(gold_text)
    result["num_steps"] = len(gen_steps)
    result["gold_steps"] = len(gold_steps)

    # Agent & tool validity
    for s in gen_steps:
        result["total_agents"] += 1
        if s["agent"] in VALID_AGENTS: result["valid_agents"] += 1
        result["total_tools"] += 1
        if s["tool"] in VALID_TOOLS: result["valid_tools"] += 1

    # ── Tool-level set comparison ────────────────────────────────────
    gen_tools = [s["tool"] for s in gen_steps if s["tool"] and s["tool"] != "none"]
    gold_tools = [s["tool"] for s in gold_steps if s["tool"] and s["tool"] != "none"]
    gen_tool_set = set(gen_tools)
    gold_tool_set = set(gold_tools)

    result["tools_correct"] = list(gen_tool_set & gold_tool_set)   # in both
    result["tools_missing"] = list(gold_tool_set - gen_tool_set)   # in gold but not gen
    result["tools_extra"] = list(gen_tool_set - gold_tool_set)     # in gen but not gold
    result["tools_correct_n"] = len(result["tools_correct"])
    result["tools_missing_n"] = len(result["tools_missing"])
    result["tools_extra_n"] = len(result["tools_extra"])

    # Agent-Tool pair F1 (Jaccard)
    gen_pairs = {(s["agent"], s["tool"]) for s in gen_steps if s["agent"] and s["tool"]}
    gold_pairs = {(s["agent"], s["tool"]) for s in gold_steps if s["agent"] and s["tool"]}
    if gen_pairs or gold_pairs:
        result["agent_tool_f1"] = len(gen_pairs & gold_pairs) / len(gen_pairs | gold_pairs)

    # ── Argument evaluation ───────────────────────────────────────────
    gold_by_at = {}
    for s in gold_steps:
        key = (s["agent"], s["tool"])
        if key not in gold_by_at and s["args"]:
            gold_by_at[key] = s["args"]

    total_key_matches, total_keys = 0, 0
    total_val_matches, total_vals = 0, 0
    args_correct, args_missing, args_extra = [], [], []

    for s in gen_steps:
        key = (s["agent"], s["tool"])
        if key in gold_by_at and s["args"]:
            gold_args = gold_by_at[key]
            gen_args = s["args"]
            gold_keys = set(gold_args.keys())
            gen_keys = set(gen_args.keys())
            if gold_keys or gen_keys:
                matched = gold_keys & gen_keys
                total_key_matches += len(matched)
                total_keys += len(gold_keys | gen_keys)
                args_correct.extend(matched)
                args_missing.extend(gold_keys - gen_keys)
                args_extra.extend(gen_keys - gold_keys)
            shared_keys = gold_keys & gen_keys
            for k in shared_keys:
                total_vals += 1
                gv = str(gold_args[k]).strip().lower()
                ev = str(gen_args.get(k, "")).strip().lower()
                if gv == ev or ("{step_" in gv and "{step_" in ev):
                    total_val_matches += 1

    result["arg_key_f1"] = total_key_matches / total_keys if total_keys else 0.0
    result["arg_value_match"] = total_val_matches / total_vals if total_vals else 0.0
    result["args_correct"] = args_correct
    result["args_missing"] = args_missing   # in gold but model forgot
    result["args_extra"] = args_extra       # model hallucinated
    result["args_correct_n"] = len(args_correct)
    result["args_missing_n"] = len(args_missing)
    result["args_extra_n"] = len(args_extra)

    # ── Step count ────────────────────────────────────────────────────
    if result["gold_steps"] > 0:
        result["step_match"] = 1.0 if result["num_steps"] == result["gold_steps"] else 0.0
        result["step_ratio"] = result["num_steps"] / result["gold_steps"]
    else:
        result["step_match"] = 0.0
        result["step_ratio"] = 0.0

    # ROUGE-L
    try:
        r = rouge_metric.compute(predictions=[gen_text], references=[gold_text])
        result["rouge_l"] = r["rougeL"]
    except Exception:
        pass
    return result


def run_evaluation(model, tokenizer, scenarios, prompt_template, mode_name, tool_descriptions=""):
    results = []
    for sc in tqdm(scenarios, desc=f"Eval ({mode_name})"):
        if "{tool_descriptions}" in prompt_template:
            prompt = prompt_template.format(tool_descriptions=tool_descriptions, question=sc["question"])
        else:
            prompt = prompt_template.format(question=sc["question"])
        chat = [{"role": "user", "content": prompt}]
        tokenized = tokenizer.apply_chat_template(chat, return_tensors="pt", add_generation_prompt=True, return_dict=True)
        input_ids = tokenized["input_ids"].to(model.device)
        attention_mask = tokenized["attention_mask"].to(model.device)
        input_len = input_ids.shape[1]
        with torch.no_grad():
            output_ids = model.generate(input_ids=input_ids, attention_mask=attention_mask,
                                        max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
                                        top_p=TOP_P, do_sample=True,
                                        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        generated = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
        metrics = evaluate_plan(generated, sc["gold_plan"])
        metrics.update({"id": sc["id"], "question": sc["question"], "type": sc.get("type", ""),
                        "generated": generated[:2000], "input_tokens": input_len,
                        "output_tokens": output_ids.shape[1] - input_len, "mode": mode_name})
        results.append(metrics)
    return results, summarize_results(results, mode_name)


def summarize_results(results, mode_name):
    n = len(results)
    if n == 0: return {"mode": mode_name, "total": 0}
    fmt_ok = sum(1 for r in results if r["has_plan_format"])
    return {
        "mode": mode_name, "total": n,
        "format_valid": fmt_ok, "format_valid_pct": 100 * fmt_ok / n,
        "avg_agent_tool_f1": np.mean([r["agent_tool_f1"] for r in results]),
        "avg_arg_key_f1": np.mean([r.get("arg_key_f1", 0) for r in results]),
        "avg_arg_value_match": np.mean([r.get("arg_value_match", 0) for r in results]),
        "avg_rouge_l": np.mean([r["rouge_l"] for r in results]),
        "avg_steps": np.mean([r["num_steps"] for r in results]),
        "avg_input_tokens": np.mean([r.get("input_tokens", 0) for r in results]),
        "avg_output_tokens": np.mean([r.get("output_tokens", 0) for r in results]),
        "agent_correctness": sum(r["valid_agents"] for r in results) / max(sum(r["total_agents"] for r in results), 1),
        "tool_correctness": sum(r["valid_tools"] for r in results) / max(sum(r["total_tools"] for r in results), 1),
        "step_exact_match": np.mean([r.get("step_match", 0) for r in results]),
        "avg_step_ratio": np.mean([r.get("step_ratio", 0) for r in results]),
        # Counts
        "total_tools_correct": sum(r.get("tools_correct_n", 0) for r in results),
        "total_tools_missing": sum(r.get("tools_missing_n", 0) for r in results),
        "total_tools_extra": sum(r.get("tools_extra_n", 0) for r in results),
        "total_args_correct": sum(r.get("args_correct_n", 0) for r in results),
        "total_args_missing": sum(r.get("args_missing_n", 0) for r in results),
        "total_args_extra": sum(r.get("args_extra_n", 0) for r in results),
    }


def print_summary(s):
    print(f"\n{'='*55}")
    print(f"  {s['mode']} — {s['total']} scenarios")
    print(f"{'='*55}")
    print(f"  Format valid:    {s.get('format_valid',0)}/{s['total']} ({s.get('format_valid_pct',0):.1f}%)")
    print(f"  Agent-Tool F1:   {s.get('avg_agent_tool_f1',0):.3f}")
    print(f"  Arg key F1:      {s.get('avg_arg_key_f1',0):.3f}")
    print(f"  Arg value match: {s.get('avg_arg_value_match',0):.3f}")
    print(f"  ROUGE-L:         {s.get('avg_rouge_l',0):.3f}")
    print(f"  Agent correct:   {s.get('agent_correctness',0):.1%}")
    print(f"  Tool correct:    {s.get('tool_correctness',0):.1%}")
    print(f"  Step exact match: {s.get('step_exact_match',0):.1%}")
    print(f"  Avg step ratio:  {s.get('avg_step_ratio',0):.2f}x (1.0=perfect)")
    print(f"  Avg steps:       {s.get('avg_steps',0):.1f}")
    print(f"  Avg tokens in:   {s.get('avg_input_tokens',0):.0f}")
    tc = s.get('total_tools_correct',0)
    tm = s.get('total_tools_missing',0)
    te = s.get('total_tools_extra',0)
    print(f"  Tools:  {tc} correct, {tm} missing, {te} extra (hallucinated)")
    ac = s.get('total_args_correct',0)
    am = s.get('total_args_missing',0)
    ae = s.get('total_args_extra',0)
    print(f"  Args:   {ac} correct, {am} missing, {ae} extra (hallucinated)")

## 5. Load Base Model CHECKED

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import wandb, time, torch

wandb.login()

if wandb.run is not None:
    wandb.finish()

wandb.init(
    project="hpml-group20-assetops",
    name=f"{MODEL_ID.split('/')[-1]}-{int(time.time())}",
    reinit=True
)

if LOAD_IN_8BIT:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    quant_label = "8-bit"
else:
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=LOAD_IN_8BIT,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    quant_label = "4-bit NF4"

print(f"Loading {MODEL_ID} in {quant_label}...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
    attn_implementation="eager"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Loaded in {time.time()-t0:.1f}s, {torch.cuda.max_memory_allocated()/1e9:.1f} GB, {sum(p.numel() for p in model.parameters())/1e9:.1f}B params")

load_time = time.time() - t0
gpu_mem = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0
num_params = sum(p.numel() for p in model.parameters())/1e9

wandb.config.update({
    "model_id": MODEL_ID,
    "quantization": quant_label,
    "load_in_8bit": LOAD_IN_8BIT,
    "num_params_B": num_params,
})

wandb.log({
    "system/load_time_sec": load_time,
    "system/gpu_mem_allocated_gb": gpu_mem,
})

wandb.finish()

## 6. Baseline Evaluation (No Fine-Tuning) CHECKED

Test the base model on held-out scenarios in two modes:
- **Informed** — full tool descriptions in the prompt (the "easy" mode)
- **Blind** — no tool descriptions (the "hard" mode, what fine-tuning must fix)

In [ ]:
import wandb

# ── INIT ─────────────────────────────
wandb.init(
    project="your-project-name",
    name="baseline_eval",
    config={
        "model": str(model),
        "num_scenarios": len(eval_scenarios),
    }
)

print(f"Running baseline evaluation on {len(eval_scenarios)} scenarios...")

baseline_informed_results, baseline_informed_summary = run_evaluation(
    model, tokenizer, eval_scenarios, INFORMED_PROMPT, "Baseline: Informed",
    tool_descriptions=TOOL_DESCRIPTIONS)

baseline_blind_results, baseline_blind_summary = run_evaluation(
    model, tokenizer, eval_scenarios, BLIND_PROMPT, "Baseline: Blind")

# ── Token overhead ─────────────────────────────
denom = baseline_informed_summary["avg_input_tokens"] or 1
token_overhead = (
    baseline_informed_summary["avg_input_tokens"]
    - baseline_blind_summary["avg_input_tokens"]
)

# ── Create comparison table ─────────────────────
import pandas as pd

df = pd.DataFrame({
    "scenario": [r["id"] for r in baseline_informed_results],
    "model": ["baseline"] * len(baseline_informed_results),

    "informed_f1": [r["agent_tool_f1"] for r in baseline_informed_results],
    "blind_f1": [r["agent_tool_f1"] for r in baseline_blind_results],

    "informed_tokens": [r["input_tokens"] for r in baseline_informed_results],
    "blind_tokens": [r["input_tokens"] for r in baseline_blind_results],

    "token_diff": [
        inf["input_tokens"] - bl["input_tokens"]
        for inf, bl in zip(baseline_informed_results, baseline_blind_results)
    ],
})

# ── LOG ─────────────────────────────
wandb.log({
    "baseline_informed/avg_input_tokens": baseline_informed_summary["avg_input_tokens"],
    "baseline_informed/agent_tool_f1": baseline_informed_summary["avg_agent_tool_f1"],
    "baseline_informed/accuracy": baseline_informed_summary.get("accuracy", 0),

    "baseline_blind/avg_input_tokens": baseline_blind_summary["avg_input_tokens"],
    "baseline_blind/agent_tool_f1": baseline_blind_summary["avg_agent_tool_f1"],
    "baseline_blind/accuracy": baseline_blind_summary.get("accuracy", 0),

    "metrics/token_overhead": token_overhead,
    "metrics/token_overhead_pct": 100 * token_overhead / denom,

    "tables/baseline_comparison": wandb.Table(dataframe=df),
})

# ── FINISH ──────────────────────────
wandb.finish()

In [ ]:

tags = ["light-mode"] if LIGHT_MODE else ["full-run"]

wandb.init(
    project="hpm1-group20-assetops",
    name="baseline-light" if LIGHT_MODE else "baseline-full",
    tags=tags,
)

wandb.config.update({
    "model": MODEL_ID,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "grad_accum": GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dataset_cap": MAX_TRAIN_EXAMPLES,
    "light_mode": LIGHT_MODE,
})

## 7. QLoRA Setup & Training Helper CHECKED

We define a reusable `train_model()` function so we can train both the generalist and specialist models.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType, PeftModel
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

def setup_lora(base_model):
    """Prepare a model for QLoRA training. Returns a new PEFT model."""
    m = prepare_model_for_kbit_training(base_model)
    lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                              target_modules=LORA_TARGET_MODULES, bias="none", task_type=TaskType.CAUSAL_LM)
    m = get_peft_model(m, lora_config)
    t, total = m.get_nb_trainable_parameters()
    print(f"Trainable: {t:,} / {total:,} ({100*t/total:.2f}%)")
    return m


def train_model(peft_model, train_data, eval_data, output_dir, epochs=NUM_EPOCHS):
    """Train a PEFT model on given data. Returns trainer."""
    os.makedirs(output_dir, exist_ok=True)
    train_ds = Dataset.from_list(train_data)
    eval_ds = Dataset.from_list(eval_data) if eval_data else None

    total_steps = len(train_ds) * epochs // (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
    warmup_steps = int(total_steps * WARMUP_RATIO)

    sft_kwargs = dict(
        output_dir=output_dir, num_train_epochs=epochs,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE, lr_scheduler_type=LR_SCHEDULER,
        warmup_steps=warmup_steps, weight_decay=WEIGHT_DECAY, bf16=BF16,
        logging_steps=10, eval_strategy="steps" if eval_ds else "no",
        eval_steps=50 if eval_ds else None,
        save_strategy="steps", save_steps=100, save_total_limit=2,
        load_best_model_at_end=bool(eval_ds), report_to="none", seed=SEED,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
    )
    # Handle max_seq_length version differences
    sft_config_params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in sft_config_params:
        sft_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
    sft_config = SFTConfig(**sft_kwargs)

    trainer_kwargs = dict(model=peft_model, args=sft_config, train_dataset=train_ds,
                          processing_class=tokenizer)
    if eval_ds: trainer_kwargs["eval_dataset"] = eval_ds
    trainer_params = inspect.signature(SFTTrainer.__init__).parameters
    if "max_seq_length" in trainer_params and "max_seq_length" not in sft_kwargs:
        trainer_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

    trainer = SFTTrainer(**trainer_kwargs)
    print(f"Training: {len(train_ds)} examples, {epochs} epochs, ~{total_steps} steps")
    t0 = time.time()
    trainer.train()
    print(f"Done in {(time.time()-t0)/60:.1f} min, final loss: {trainer.state.log_history[-1].get('train_loss', trainer.state.log_history[-1].get('loss', '?'))}")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer

print("Training helper ready.")

## 8. Prepare Training Data — 3-Tier Architecture CHECKED

We train three types of models:

1. **Generalist** — one model trained on everything (baseline for comparison)
2. **Planner** — specialized in routing: given a question, output which agents/tools/dependencies to use (no args)
3. **Specialists** (per agent) — given a step's task + agent + tool, output the correct arguments

```
Question → [Planner] → routing plan → [Specialist per step] → full plan with args
```

This way, the planner focuses on *what to do* and specialists focus on *how to do it*.

In [ ]:
def get_agents_in_plan(plan_text):
    return [a for a in re.findall(r'#Agent\d+:\s*(\S+)', plan_text) if a not in ('none', '')]

def strip_plan_to_routing(plan_text):
    """Strip a plan to routing-only (Task/Agent/Tool/Dependency). Remove Args/ExpectedOutput."""
    lines = []
    for line in plan_text.split("\n"):
        stripped = line.strip()
        if any(stripped.startswith(f"#{tag}") for tag in ["Task", "Agent", "Tool", "Dependency"]):
            if not stripped.startswith("#Args") and not stripped.startswith("#ExpectedOutput"):
                lines.append(line)
        elif stripped == "":
            lines.append("")
    return "\n".join(lines).strip()

def extract_specialist_steps(plan_text):
    """Extract per-step specialist examples: input=(task, agent, tool) -> output=(args, expected)."""
    steps = []
    for block in re.split(r'(?=#Task\d+:)', plan_text):
        block = block.strip()
        if not block: continue
        task_m = re.search(r'#Task\d+:\s*(.*)', block)
        agent_m = re.search(r'#Agent\d+:\s*(\S+)', block)
        tool_m = re.search(r'#Tool\d+:\s*(\S+)', block)
        args_m = re.search(r'#Args\d+:\s*(.*)', block)
        exp_m = re.search(r'#ExpectedOutput\d+:\s*(.*)', block)
        dep_m = re.search(r'#Dependency\d+:\s*(.*)', block)
        if task_m and agent_m and tool_m:
            agent = agent_m.group(1).strip()
            if agent in ('none', ''): continue
            instruction = f"Generate the arguments for this tool call:\nTask: {task_m.group(1).strip()}\nAgent: {agent}\nTool: {tool_m.group(1).strip()}\nDependency: {dep_m.group(1).strip() if dep_m else 'None'}"
            response = f"#Args: {args_m.group(1).strip() if args_m else '{}'}\n#ExpectedOutput: {exp_m.group(1).strip() if exp_m else 'Result'}"
            steps.append({"agent": agent, "instruction": instruction, "response": response})
    return steps

# ── 1. Generalist data (everything) ──────────────────────────────
all_train = [{"messages": d["messages"]} for d in clean_tool + clean_plan + clean_exec]
random.shuffle(all_train)
if MAX_TRAIN_EXAMPLES:
    all_train = all_train[:MAX_TRAIN_EXAMPLES]
sp = int(len(all_train) * 0.95)
generalist_train, generalist_eval = all_train[:sp], all_train[sp:]

# ── 2. Planner data (routing-only plans) ─────────────────────────
planner_data = []
for d in clean_plan:
    if d.get("metadata", {}).get("category") != "planning": continue
    routing = strip_plan_to_routing(d["messages"][1]["content"])
    if "#Task" in routing and "#Agent" in routing:
        planner_data.append({"messages": [
            {"role": "user", "content": d["messages"][0]["content"]},
            {"role": "assistant", "content": routing},
        ]})
# Also add tool knowledge (helps planner learn agent/tool associations)
for d in clean_tool:
    planner_data.append({"messages": d["messages"]})

random.shuffle(planner_data)
if MAX_TRAIN_EXAMPLES:
    planner_data = planner_data[:MAX_TRAIN_EXAMPLES]
sp = int(len(planner_data) * 0.95)
planner_train, planner_eval = planner_data[:sp], planner_data[sp:]

# ── 3. Specialist data (per-agent step-level arg generation) ─────
specialist_data = defaultdict(list)
for d in clean_plan:
    if d.get("metadata", {}).get("category") != "planning": continue
    for step in extract_specialist_steps(d["messages"][1]["content"]):
        specialist_data[step["agent"]].append({"messages": [
            {"role": "user", "content": step["instruction"]},
            {"role": "assistant", "content": step["response"]},
        ]})

print(f"Training data prepared:")
print(f"  Generalist:  {len(generalist_train)} train, {len(generalist_eval)} eval")
print(f"  Planner:     {len(planner_train)} train, {len(planner_eval)} eval")
print(f"  Specialists:")
for agent, data in sorted(specialist_data.items()):
    print(f"    {agent}: {len(data)} step-level examples")

## 9. Train Generalist Model CHECKED

One model trained on ALL data — the baseline for comparison.

In [ ]:
import matplotlib.pyplot as plt

wandb.init(
    project="hpm1-group20-assetops",
    name=f"generalist-{'light' if LIGHT_MODE else 'full'}",
    tags=["generalist", "light-mode" if LIGHT_MODE else "full-run"],
    reinit=True,
)

wandb.config.update({
    "model": MODEL_ID,
    "task": "generalist",
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "grad_accum": GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_size": len(generalist_train),
    "eval_size": len(generalist_eval),
})

generalist_dir = f"{OUTPUT_DIR}/generalist"

model_generalist = load_model(MODEL_ID)
peft_generalist = setup_lora(model_generalist)

gen_trainer = train_model(
    peft_generalist,
    generalist_train,
    generalist_eval,
    generalist_dir
)

log_h = gen_trainer.state.log_history

train_loss = [(h["step"], h["loss"]) for h in log_h if "loss" in h]
eval_loss = [(h["step"], h["eval_loss"]) for h in log_h if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(10, 4))

if train_loss:
    ax.plot(*zip(*train_loss), label="Train", alpha=0.7)

if eval_loss:
    ax.plot(*zip(*eval_loss), label="Eval", linewidth=2)

ax.set(
    xlabel="Step",
    ylabel="Loss",
    title="Generalist Training"
)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

wandb.log({
    "plots/generalist_loss_curve": wandb.Image(fig)
})

if train_loss:
    wandb.summary["final/train_loss"] = train_loss[-1][1]
if eval_loss:
    wandb.summary["final/eval_loss"] = eval_loss[-1][1]


## 10. Evaluate Generalist CHECKED

In [ ]:
# ── Run evaluation ───────────────────────────────────────
gen_blind_results, gen_blind_summary = run_evaluation(
    peft_generalist, tokenizer, eval_scenarios,
    BLIND_PROMPT, "Generalist: Blind"
)
print_summary(gen_blind_summary)

gen_informed_results, gen_informed_summary = run_evaluation(
    peft_generalist, tokenizer, eval_scenarios,
    INFORMED_PROMPT, "Generalist: Informed",
    tool_descriptions=TOOL_DESCRIPTIONS
)
print_summary(gen_informed_summary)

# ── Token overhead ───────────────────────────────────────
denom = gen_informed_summary["avg_input_tokens"] or 1
token_overhead = (
    gen_informed_summary["avg_input_tokens"]
    - gen_blind_summary["avg_input_tokens"]
)

print(
    f"\nToken overhead: {token_overhead:.0f} tokens "
    f"({100 * token_overhead / denom:.0f}% of informed prompt)"
)

# ── Show examples ─────────────────────────────────────────
for r in gen_blind_results[:3]:
    print(f"\n--- ID {r['id']} ({r['type']}): {r['question'][:55]}... ---")
    print(
        f"  AT-F1={r['agent_tool_f1']:.2f} "
        f"ROUGE={r['rouge_l']:.2f} "
        f"Steps={r['num_steps']}(gold:{r['gold_steps']})"
    )
    print(f"  {r['generated'][:200]}")

# ── Create W&B table ─────────────────────────────────────
df = pd.DataFrame({
    "scenario": [r["id"] for r in gen_informed_results],
    "model": ["generalist"] * len(gen_informed_results),

    "informed_f1": [r["agent_tool_f1"] for r in gen_informed_results],
    "blind_f1": [r["agent_tool_f1"] for r in gen_blind_results],

    "informed_tokens": [r["input_tokens"] for r in gen_informed_results],
    "blind_tokens": [r["input_tokens"] for r in gen_blind_results],

    "rouge_l": [r["rouge_l"] for r in gen_informed_results],
    "num_steps": [r["num_steps"] for r in gen_informed_results],
})

# ── Log to W&B ───────────────────────────────────────────
wandb.log({
    # Summary metrics
    "generalist_informed/avg_input_tokens": gen_informed_summary["avg_input_tokens"],
    "generalist_informed/agent_tool_f1": gen_informed_summary["avg_agent_tool_f1"],
    "generalist_informed/accuracy": gen_informed_summary.get("accuracy", 0),

    "generalist_blind/avg_input_tokens": gen_blind_summary["avg_input_tokens"],
    "generalist_blind/agent_tool_f1": gen_blind_summary["avg_agent_tool_f1"],
    "generalist_blind/accuracy": gen_blind_summary.get("accuracy", 0),

    # Token efficiency
    "generalist/token_overhead": token_overhead,
    "generalist/token_overhead_pct": 100 * token_overhead / denom,

    # Table
    "tables/generalist_comparison": wandb.Table(dataframe=df),
})

# ── Optional summary ─────────────────────────────────────
wandb.summary["generalist/f1_informed"] = gen_informed_summary["avg_agent_tool_f1"]
wandb.summary["generalist/f1_blind"] = gen_blind_summary["avg_agent_tool_f1"]
wandb.summary["generalist/token_overhead"] = token_overhead

wandb.finish()

## 11. Train Planner Model CHECKED

The planner is specialized in **routing**: given a question, it outputs which agents, tools, and dependencies to use — but NOT the arguments. This is a simpler task than full plan generation.

The planner's output looks like:
```
#Task1: Retrieve assets for MAIN site
#Agent1: IoTAgent
#Tool1: assets
#Dependency1: None
```
No `#Args` or `#ExpectedOutput` — those come from the specialist.

In [ ]:
import matplotlib.pyplot as plt
import wandb

del peft_generalist
torch.cuda.empty_cache()

wandb.init(
    project="hpm1-group20-assetops",
    name=f"planner-{'light' if LIGHT_MODE else 'full'}",
    tags=["planner", "light-mode" if LIGHT_MODE else "full-run"],
    reinit=True,
)

wandb.config.update({
    "model": MODEL_ID,
    "task": "planner",
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "grad_accum": GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_size": len(planner_train),
    "eval_size": len(planner_eval),
})

planner_dir = f"{OUTPUT_DIR}/planner"

model_planner = load_model(MODEL_ID)
peft_planner = setup_lora(model_planner)

planner_trainer = train_model(
    peft_planner,
    planner_train,
    planner_eval,
    planner_dir
)

log_h = planner_trainer.state.log_history

train_loss = [(h["step"], h["loss"]) for h in log_h if "loss" in h]
eval_loss = [(h["step"], h["eval_loss"]) for h in log_h if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(10, 4))

if train_loss:
    ax.plot(*zip(*train_loss), label="Train", alpha=0.7)

if eval_loss:
    ax.plot(*zip(*eval_loss), label="Eval", linewidth=2)

ax.set(xlabel="Step", ylabel="Loss", title="Planner Training")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

wandb.log({
    "plots/planner_loss_curve": wandb.Image(fig)
})

if train_loss:
    wandb.summary["final/train_loss"] = train_loss[-1][1]
if eval_loss:
    wandb.summary["final/eval_loss"] = eval_loss[-1][1]

## 12. Evaluate Planner (Routing Accuracy) CHECKED

In [ ]:
import pandas as pd

planner_blind_results, planner_blind_summary = run_evaluation(
    peft_planner, tokenizer, eval_scenarios,
    BLIND_PROMPT, "Planner: Blind"
)

print_summary(planner_blind_summary)

for r in planner_blind_results[:3]:
    print(f"\n--- ID {r['id']}: {r['question'][:55]}... ---")
    print(f"  AT-F1={r['agent_tool_f1']:.2f}, "
          f"Steps={r['num_steps']}(gold:{r['gold_steps']})")
    print(f"  {r['generated'][:250]}")

df = pd.DataFrame({
    "scenario": [r["id"] for r in planner_blind_results],
    "model": ["planner"] * len(planner_blind_results),
    "agent_tool_f1": [r["agent_tool_f1"] for r in planner_blind_results],
    "num_steps": [r["num_steps"] for r in planner_blind_results],
    "gold_steps": [r["gold_steps"] for r in planner_blind_results],
    "input_tokens": [r["input_tokens"] for r in planner_blind_results],
})

wandb.log({
    "planner/avg_input_tokens": planner_blind_summary["avg_input_tokens"],
    "planner/agent_tool_f1": planner_blind_summary["avg_agent_tool_f1"],
    "planner/accuracy": planner_blind_summary.get("accuracy", 0),
    "tables/planner_results": wandb.Table(dataframe=df),
})

wandb.summary["planner/f1"] = planner_blind_summary["avg_agent_tool_f1"]
wandb.summary["planner/avg_tokens"] = planner_blind_summary["avg_input_tokens"]

wandb.finish()

## 13. Train Specialist Models (Per-Agent Arg Generation) CHECKED

Each specialist is trained on step-level examples: given a task description + agent + tool name, it generates the correct arguments.

In production: Planner outputs the routing → each step is sent to the matching specialist → specialist fills in args.

In [ ]:
import wandb
import torch
import random
import pandas as pd

wandb.init(
    project="hpm1-group20-assetops",
    name=f"planner-specialists-{'light' if LIGHT_MODE else 'full'}",
    tags=["planner", "specialists", "light-mode" if LIGHT_MODE else "full-run"],
    reinit=True,
)

wandb.config.update({
    "model": MODEL_ID,
    "task": "planner+specialists",
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "grad_accum": GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "num_specialist_types": len(specialist_data),
})

if "peft_generalist" in globals():
    del peft_generalist
torch.cuda.empty_cache()

planner_dir = f"{OUTPUT_DIR}/planner"

model_planner = load_model(MODEL_ID)
peft_planner = setup_lora(model_planner)

planner_trainer = train_model(
    peft_planner,
    planner_train,
    planner_eval,
    planner_dir
)

planner_log_h = planner_trainer.state.log_history
planner_tl = [(h["step"], h["loss"]) for h in planner_log_h if "loss" in h]
planner_el = [(h["step"], h["eval_loss"]) for h in planner_log_h if "eval_loss" in h]

planner_train_loss = planner_tl[-1][1] if planner_tl else None
planner_eval_loss = planner_el[-1][1] if planner_el else None

wandb.log({
    "planner/train_loss": planner_train_loss,
    "planner/eval_loss": planner_eval_loss,
})

del peft_planner
torch.cuda.empty_cache()

specialist_models = {}
specialist_summary = []

for agent_name, data in sorted(specialist_data.items()):
    if len(data) < 20:
        continue

    random.shuffle(data)
    if MAX_TRAIN_EXAMPLES:
        data = data[:min(len(data), MAX_TRAIN_EXAMPLES)]

    sp = max(1, int(len(data) * 0.9)) if len(data) > 1 else len(data)

    train_split = data[:sp]
    eval_split = data[sp:] if sp < len(data) else None

    model_spec = load_model(MODEL_ID)
    spec_model = setup_lora(model_spec)

    spec_dir = f"{OUTPUT_DIR}/specialist_{agent_name}"

    spec_trainer = train_model(
        spec_model,
        train_split,
        eval_split,
        spec_dir
    )

    log_h = spec_trainer.state.log_history
    tl = [(h["step"], h["loss"]) for h in log_h if "loss" in h]
    el = [(h["step"], h["eval_loss"]) for h in log_h if "eval_loss" in h]

    train_loss = tl[-1][1] if tl else None
    eval_loss = el[-1][1] if el else None

    wandb.log({
        f"specialist/{agent_name}/train_loss": train_loss,
        f"specialist/{agent_name}/eval_loss": eval_loss,
    })

    specialist_summary.append({
        "agent": agent_name,
        "train_loss": train_loss,
        "eval_loss": eval_loss,
        "num_examples": len(data)
    })

    specialist_models[agent_name] = spec_model

df = pd.DataFrame(specialist_summary)

wandb.log({
    "tables/specialists_summary": wandb.Table(dataframe=df),
    "num_specialists_trained": len(specialist_models),
})

wandb.summary["planner/train_loss"] = planner_train_loss
wandb.summary["planner/eval_loss"] = planner_eval_loss
wandb.summary["num_specialists_trained"] = len(specialist_models)

## 14. Evaluate Planner + Specialist Pipeline CHECKED

The full pipeline: Planner generates routing → each step is sent to the matching specialist for arg generation → combine into final plan.

We evaluate the *combined output* against the gold plan.

In [ ]:
pipeline_results = []

for sc in tqdm(eval_scenarios, desc="Eval (Planner+Specialist pipeline)"):
    planner_r = [r for r in planner_blind_results if r["id"] == sc["id"]]
    if not planner_r:
        continue

    planner_output = planner_r[0]["generated"]
    planner_steps = parse_plan(planner_output)

    full_plan_lines = []

    for step in planner_steps:
        agent = step["agent"]
        tool = step["tool"]

        if agent in specialist_models:
            spec = specialist_models[agent]

            spec_prompt = (
                f"Generate the arguments for this tool call:\n"
                f"Task: {step['task']}\n"
                f"Agent: {agent}\n"
                f"Tool: {tool}\n"
                f"Dependency: None"
            )

            chat = [{"role": "user", "content": spec_prompt}]
            tokenized = tokenizer.apply_chat_template(
                chat,
                return_tensors="pt",
                add_generation_prompt=True,
                return_dict=True
            )

            input_ids = tokenized["input_ids"].to(spec.device)
            attention_mask = tokenized["attention_mask"].to(spec.device)

            with torch.no_grad():
                out = spec.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=256,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    do_sample=True,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
                )

            spec_output = tokenizer.decode(
                out[0][input_ids.shape[1]:],
                skip_special_tokens=True
            )

            args_m = re.search(r'#Args:\s*(.*)', spec_output)
            exp_m = re.search(r'#ExpectedOutput:\s*(.*)', spec_output)

            args = args_m.group(1).strip() if args_m else "{}"
            expected = exp_m.group(1).strip() if exp_m else ""

        else:
            args, expected = "{}", ""

        full_plan_lines.append(f"#Task{step['step']}: {step['task']}")
        full_plan_lines.append(f"#Agent{step['step']}: {agent}")
        full_plan_lines.append(f"#Tool{step['step']}: {tool}")
        full_plan_lines.append(f"#Args{step['step']}: {args}")
        full_plan_lines.append(f"#Dependency{step['step']}: None")

        if expected:
            full_plan_lines.append(f"#ExpectedOutput{step['step']}: {expected}")

        full_plan_lines.append("")

    combined_plan = "\n".join(full_plan_lines)

    metrics = evaluate_plan(combined_plan, sc["gold_plan"])

    metrics.update({
        "id": sc["id"],
        "question": sc["question"],
        "type": sc.get("type", ""),
        "generated": combined_plan[:2000],
        "mode": "pipeline",
    })

    pipeline_results.append(metrics)

pipeline_summary = summarize_results(pipeline_results, "Pipeline: Planner+Specialist")
print_summary(pipeline_summary)

df = pd.DataFrame({
    "scenario": [r["id"] for r in pipeline_results],
    "model": ["pipeline"] * len(pipeline_results),
    "agent_tool_f1": [r["agent_tool_f1"] for r in pipeline_results],
    "rouge_l": [r["rouge_l"] for r in pipeline_results],
    "num_steps": [r["num_steps"] for r in pipeline_results],
    "type": [r.get("type", "") for r in pipeline_results],
})

wandb.log({
    "pipeline/agent_tool_f1": pipeline_summary["avg_agent_tool_f1"],
    "pipeline/accuracy": pipeline_summary.get("accuracy", 0),
    "pipeline/rouge_l": pipeline_summary.get("avg_rouge_l", 0),
    "tables/pipeline_results": wandb.Table(dataframe=df),
})

wandb.summary["pipeline/f1"] = pipeline_summary["avg_agent_tool_f1"]
wandb.summary["pipeline/accuracy"] = pipeline_summary.get("accuracy", 0]

wandb.finish()

## 15. Full Comparison — All Approaches CHECKED

In [ ]:
wandb.init(
    project="hpm1-group20-assetops",
    name="final-comparison",
    tags=["comparison", "final"],
    reinit=True,
)

all_summaries = [
    baseline_informed_summary,
    baseline_blind_summary,
    gen_informed_summary,
    gen_blind_summary,
    planner_blind_summary,
    pipeline_summary,
]

import pandas as pd

comp = pd.DataFrame(all_summaries)

display_cols = {
    "mode": "Mode",
    "format_valid_pct": "Format%",
    "avg_agent_tool_f1": "AT-F1",
    "avg_arg_key_f1": "ArgKey",
    "avg_arg_value_match": "ArgVal",
    "step_exact_match": "StepEM",
    "avg_step_ratio": "StepR",
    "avg_rouge_l": "ROUGE",
    "agent_correctness": "Agent%",
    "tool_correctness": "Tool%",
    "avg_steps": "Steps",
    "avg_input_tokens": "TokIn",
}

# Keep only available columns
comp = comp[[c for c in display_cols if c in comp.columns]]
comp.columns = [display_cols[c] for c in comp.columns]

# ── Print (keep this) ────────────────────────────────────
print("\n" + "=" * 110)
print("  FULL COMPARISON: 6 Approaches")
print("=" * 110)
print(comp.to_string(index=False, float_format="%.3f"))

print(f"\nKey results:")
print(f"  Baseline blind AT-F1:         {baseline_blind_summary['avg_agent_tool_f1']:.3f}")
print(f"  Generalist blind AT-F1:       {gen_blind_summary['avg_agent_tool_f1']:.3f}")
print(f"  Planner-only blind AT-F1:     {planner_blind_summary['avg_agent_tool_f1']:.3f}")
print(f"  Planner+Specialist AT-F1:     {pipeline_summary['avg_agent_tool_f1']:.3f}")

# ── Log FULL comparison table (CRITICAL) ─────────────────
wandb.log({
    "tables/final_comparison": wandb.Table(dataframe=comp)
})

# ── Log key headline metrics (VERY IMPORTANT) ────────────
wandb.log({
    "final/baseline_blind_f1": baseline_blind_summary["avg_agent_tool_f1"],
    "final/generalist_blind_f1": gen_blind_summary["avg_agent_tool_f1"],
    "final/planner_blind_f1": planner_blind_summary["avg_agent_tool_f1"],
    "final/pipeline_f1": pipeline_summary["avg_agent_tool_f1"],
})

# ── Optional: summary panel (nice in W&B UI) ─────────────
wandb.summary["best_model"] = "pipeline"
wandb.summary["pipeline_f1"] = pipeline_summary["avg_agent_tool_f1"]
wandb.summary["generalist_f1"] = gen_blind_summary["avg_agent_tool_f1"]
wandb.summary["baseline_f1"] = baseline_blind_summary["avg_agent_tool_f1"]

wandb.finish()

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
modes = ["Base\nInformed", "Base\nBlind", "Gen\nInformed", "Gen\nBlind", "Planner\nBlind", "Pipeline\nP+S"]
colors = ["#3b82f6", "#ef4444", "#22c55e", "#f59e0b", "#8b5cf6", "#ec4899"]

for ax, metric, title in [
    (axes[0], "format_valid_pct", "Format Valid (%)"),
    (axes[1], "avg_agent_tool_f1", "Agent-Tool F1"),
    (axes[2], "agent_correctness", "Agent Correctness"),
]:
    vals = [s.get(metric, 0) for s in all_summaries]
    ax.bar(modes, vals, color=colors)
    ax.set_title(title, fontsize=11)
    for i, v in enumerate(vals):
        fmt = f"{v:.0f}%" if "pct" in metric else f"{v:.3f}"
        ax.text(i, v + max(vals)*0.02, fmt, ha="center", fontweight="bold", fontsize=7)

plt.suptitle(f"AssetOpsBench: {MODEL_ID} — All Approaches Compared", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/full_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 16. Per-Domain Analysis CHECKED

In [ ]:
wandb.init(
    project="hpm1-group20-assetops",
    name="per-type-analysis",
    tags=["analysis", "per-type"],
    reinit=True,
)

gen_by_id = {r["id"]: r for r in gen_blind_results}
plan_by_id = {r["id"]: r for r in planner_blind_results}
pipe_by_id = {r["id"]: r for r in pipeline_results}

rows = []
for sc in eval_scenarios:
    sid = sc["id"]
    rows.append({
        "id": sid,
        "type": sc["type"],

        "gen_atf1": gen_by_id.get(sid, {}).get("agent_tool_f1", 0),
        "planner_atf1": plan_by_id.get(sid, {}).get("agent_tool_f1", 0),
        "pipeline_atf1": pipe_by_id.get(sid, {}).get("agent_tool_f1", 0),

        "gen_rouge": gen_by_id.get(sid, {}).get("rouge_l", 0),
        "pipeline_rouge": pipe_by_id.get(sid, {}).get("rouge_l", 0),
    })

import pandas as pd

df = pd.DataFrame(rows)

# ── Per-type aggregation ────────────────────────────────
type_comp = df.groupby("type").agg(
    gen_atf1=("gen_atf1", "mean"),
    planner_atf1=("planner_atf1", "mean"),
    pipeline_atf1=("pipeline_atf1", "mean"),
    count=("id", "count"),
).round(3)

print("Per-type comparison:")
print(type_comp.to_string())

# ── Winner per type ─────────────────────────────────────
winner_rows = []
for t in type_comp.index:
    best_col = type_comp.loc[t, ["gen_atf1", "planner_atf1", "pipeline_atf1"]].idxmax()
    best_val = type_comp.loc[t, best_col]

    print(f"  {t}: best={best_col} ({best_val:.3f})")

    winner_rows.append({
        "type": t,
        "best_model": best_col,
        "best_score": best_val
    })

winner_df = pd.DataFrame(winner_rows)

# ── Log to W&B ──────────────────────────────────────────
wandb.log({
    "tables/per_scenario_results": wandb.Table(dataframe=df),
    "tables/per_type_comparison": wandb.Table(dataframe=type_comp.reset_index()),
    "tables/per_type_winners": wandb.Table(dataframe=winner_df),
})

# ── Optional summary insights ────────────────────────────
wandb.summary["num_types"] = len(type_comp)
wandb.summary["pipeline_avg_f1_by_type"] = type_comp["pipeline_atf1"].mean()

wandb.finish()

## 17. Token Savings CHECKED

In [ ]:
wandb.init(name="token-analysis", reinit=True)

informed_tok = baseline_informed_summary["avg_input_tokens"]
blind_tok = gen_blind_summary["avg_input_tokens"]

denom = informed_tok or 1
savings = informed_tok - blind_tok
savings_pct = 100 * savings / denom

print("=" * 60)
print("  TOKEN SAVINGS ANALYSIS")
print("=" * 60)
print(f"  Informed prompt:      {informed_tok:.0f} tokens (with tool descriptions)")
print(f"  Blind prompt:         {blind_tok:.0f} tokens (no tool descriptions)")
print(f"  Savings per query:    {savings:.0f} tokens ({savings_pct:.0f}%)")
print(f"  Per 152 scenarios:    {savings*152:,.0f} tokens saved")
print(f"  Per 1000 queries/day: {savings*1000:,.0f} tokens/day")

print(f"\n  Production architecture argument:")
print(f"  - Planner model handles routing (small, fast)")
print(f"  - Specialist models handle args (domain-specific, parallelizable)")
print(f"  - Combined latency can be LOWER than one generalist pass")
print(f"  - Each specialist stays small and focused")

# ── Log to W&B ──────────────────────────────────────────
wandb.log({
    "tokens/informed_avg": informed_tok,
    "tokens/blind_avg": blind_tok,
    "tokens/savings_per_query": savings,
    "tokens/savings_pct": savings_pct,

    "tokens/savings_per_152": savings * 152,
    "tokens/savings_per_1000": savings * 1000,
})

# ── Optional: store key results in summary ───────────────
wandb.summary["token_savings_per_query"] = savings
wandb.summary["token_savings_pct"] = savings_pct
wandb.summary["daily_token_savings_1k"] = savings * 1000

# ── Optional: log explanation as text (nice touch) ───────
wandb.log({
    "notes/architecture": wandb.Html(f"""
    <b>Production Architecture Insight:</b><br>
    Planner handles routing (lightweight)<br>
    Specialists handle arguments (parallelizable)<br>
    Lower total latency vs monolithic model<br>
    Better modularity and scalability
    """)
})

wandb.finish()

## 18. Save Results CHECKED

In [ ]:
wandb.init(name="artifact-upload", reinit=True)

results = {
    "config": {"model": MODEL_ID, "lora_r": LORA_R, "epochs": NUM_EPOCHS,
               "lr": LEARNING_RATE, "held_out": NUM_HELD_OUT, "timestamp": datetime.now().isoformat()},
    "summaries": {s["mode"]: s for s in all_summaries},
    "token_savings": {"informed": informed_tok, "blind": blind_tok, "savings": savings},
    "per_scenario": rows,
}
with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
df.to_csv(f"{OUTPUT_DIR}/per_scenario.csv", index=False)
print(f"Saved to {OUTPUT_DIR}/")

print(f"\n{'='*60}")
print(f"  EXPERIMENT COMPLETE")
print(f"{'='*60}")
for s in all_summaries:
    print(f"  {s['mode']:40s} AT-F1={s.get('avg_agent_tool_f1',0):.3f} Agent={s.get('agent_correctness',0):.1%}")
print(f"\n  Token savings: {savings:.0f}/query ({100*savings/informed_tok:.0f}%)")

# ── Log artifacts to W&B ────────────────────────────────
artifact = wandb.Artifact("experiment-results", type="results")

artifact.add_file(f"{OUTPUT_DIR}/results.json")
artifact.add_file(f"{OUTPUT_DIR}/per_scenario.csv")

wandb.log_artifact(artifact)

wandb.finish()

## 19. (Optional) Download

In [ ]:
os.system("cd /content && zip -r /content/assetops_results.zip output/")
try:
    from google.colab import files
    files.download("/content/assetops_results.zip")
except ImportError:
    print("Results at /content/output/")